#### Automatic Summary Evaluation

This project uses an LLM-as-a-judge technique to evaluate summaries generated by machine learning models. Summary evaluation is useful in production environments, and has analogies in similar evaluations of GenAI outputs (or the automated evaluation of human outputs).

This notebook uses data from SummEval [2] which is a dataset of CNN and Dailymail articles, as well as machine generated summaries of those articles, and huamn evaluations of those summaries. I use a question-answer generation scheme to evaluate the summaries on consistency and similar metrics.

Typically, LLM-as-a-judge means a variant of G-Eval [1] which is technique that essentially defines a metric to the LLM in the system prompt, then asks the model to rate some piece of input text using that metric. G-Eval typically has high correlation with human evaluations and is a powerful tool. It is especially powerful when the metric is easy to define - How would you the define the "relevance" of a summary using a rules-based-nlp method? How do you get thousands of "relevance" examples to train a model?

G-Eval was originally used to better measure Relevance, Consistency, Fluency, and Coherence. Relevance being the measure of how well the summary captures the key points of the article. Consistency being the measure of whether the summary reproduces all facts accurately and does not make up untrue information. Fluency being the measure of the quality of individual sentences (gramatically correct). Coherence being the measure of the quality of the summary as a whole.

In general, machine written summaries from LLMs now perform well in Fluency and Coherence - to the point that the focus can shift nearly entirely on Relevance and Consistency. The biggest struggle for LLMs is often consistency - stemming from an incorrect representation of events in the source text or flat out hallucinations. Relevance is not as "solved" as Fluency and Coherence, but LLMs often produce output with acceptable levels of relevance.

While G-Eval is a state-of-the-art metric, this notebook opts to instead use a question-answer generation (QAG) framework. QAG-based evaluation has been shown to be more correlated with human evalitation than G-Eval for consistency and similar metrics [3]. The similar metrics are faithfulness (also known as groundedness) and coverage (also known as completeness), which are discussed further in the "Scoring the Summaries" section.

The idea behind QAG is to use the summaries to generate questions, then (without preserving the history that includes the summary) have the model answer those questions using the source text. This method can also be used to generate questions from the source text and to answer those questions using the summaries. The details of why to do one vs the other are also given in the "Scoring the Summaries" section.

This implementation uses [4] and [5] for inspiration.

In [1]:
import pandas as pd
import requests
import json
import numpy as np
import aiohttp
import asyncio

# Dataset paths
SUMM_EVAL_DATA = "./data/summEval/model_annotations.aligned.jsonl"
TEST_FULL_TEXT = "./data/summEval/test-00000-of-00001.parquet"

# LLAMA Parameters
MODEL = "llama3.2:3b"
LLAMA_URL = "http://localhost:11434/api/chat"

# Hyperparameters for question generations
#   How many characters per question? Represents the information density of our documents
SUMMARY_CHARS_PER_Q = 60
ARTICLE_CHARS_PER_Q = 120

# When generating questions, generate a few extra and truncate down to the desired amount
QUESTION_BUFFER = 2

#### Reading in the the Dataset

This dataset is from SummEval and comprised of full text articles from CNN and Dailymail, summaries generated by various models, and human evaluations of those summaries.

The goal of this notebook is to create a automated method can evaluate the faithfulness and coverage of a summary given the source article. But how will we know if we are successful?

One way is to compare the automated method's results to human evaluations on the same summary/article pair - the idea being we want to perform the task like a human would. By comparing to many such pairs, we can evaluate the method using correlation.

In [2]:
# Summaries and summary scores are available here:
#   https://storage.googleapis.com/sfr-summarization-repo-research/model_annotations.aligned.jsonl
# If you don't want to follow on a random link from me (or if that link becomes unavailable), you should be able
#   to get it through to the SummEval paper's GitHub:
#   https://github.com/Yale-LILY/SummEval?tab=readme-ov-file#human-annotations
summaries_and_labels = pd.read_json(SUMM_EVAL_DATA, lines=True)
summaries_and_labels["id"] = summaries_and_labels["id"].apply(lambda x: x.split("-")[-1])
summaries_and_labels["consistency"] = summaries_and_labels["expert_annotations"].apply(lambda x: np.mean([xi["consistency"] for xi in x]))

# The articles used in the test data of SummEval did not appear in the dataset via the download link on the SummEval
#   GitHub. I am unsure if they have been removed or if they were not intended to be made available.
# Regardless, they are HuggingFace, and confirmed that the summaries provided by SummEval match up to the articles.
#   https://huggingface.co/datasets/abisee/cnn_dailymail/tree/main/3.0.0
articles = pd.read_parquet(TEST_FULL_TEXT)

summ_eval = summaries_and_labels.merge(articles, on="id")
summ_eval.drop(["id", "filepath", "highlights", "expert_annotations", "turker_annotations", "references"], axis=1, inplace=True)
summ_eval.rename(columns={"decoded": "machine_summary"}, inplace=True)
summ_eval.to_csv("./data/summEval/summEvalTest.csv")

#### Calling the models

* post request
* Similar payloads but different APIs, so 1 function to call llama, and another to call GPT
    * both have temperature, top_p, seed, and number of output tokens

In [3]:
# Make sure to run 
#   OLLAMA_NUM_PARALLEL=3 ollama serve
async def call_llama(session, payload):
    async with session.post(LLAMA_URL, json=payload) as resp:
        response = await resp.json()
        resp.raise_for_status()
    
    if response["done_reason"] != "stop":
        return None
    
    return response["message"]["content"]

def llama_payload(system_prompt, user_message, prompt_type, num_questions=None):
    return {
        "model": MODEL,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_message
            },
        ],
        "format": "json",
        "stream": False,
        "options": {
            "seed": 314159265,
            "temperature": 0.0,
            "top_p": 0.0,
            "top_k": 1,
            "num_predict": 10 + 100*num_questions if prompt_type == "questions" else 15
        }, 
    }

#### Prompt Engineering

* json structured output (both openAI GPT and LLAMA support it) - good way to put guardrails on model output
* instructions
* one shot prompting
* system vs user message

In [4]:
# Here we define some examples for one-shot prompting
#   The summary and article examples are unrelated, but are both from the SummEval dataset
#   Neither the summary nor article are in the actual dataset used for evaluation

EXAMPLE_SUMMARY = "buckingham palace guard slipped on manhole cover in front of hundreds of horrified tourists. the queen's guard was left red-faced after he slipped on a manhole cover. he lost his footing and dropped his rifle. the incident was caught on camera. the guard is thought to have slipped because of metal shutters nailed to the soles of his boots."
EXAMPLE_SUMMARY_QUESTIONS = [
    "Did the buckingham palace guard slip on a manhole cover?",
    "Did the buckingham palace guard in front of tourists?",
    "Was the queen's guard left red-faced?",
    "Did the guard drop his rifle?",
    "Was the incident caught on camera?",
    "Is the guard thought to have slipped due to the metal shutters nailed to the soles of his boots?",
]

EXAMPLE_ARTICLE = """United Nations (CNN) -- The U.N. Security Council approved a resolution Monday to send 4,200 peacekeepers to Abyei, Sudan, as part of a recent agreement between Sudan and Southern Sudan. The resolution will establish, for six months, the United Nations Interim Security Force for Abyei (UNISFA), comprising "a maximum of 4,200 military personnel, 50 police personnel, and appropriate civilian support," the resolution states. It passed the council unanimously, 15-0. In a statement released by the State Department, Secretary Hiliary Clinton commended the swift passage of the resolution. "Abyei has been a source of regional tension for many years," the statement said. "We urge the parties to reach an immediate cease-fire and to provide aid workers with the unfettered access required to deliver humanitarian assistance to innocent civilians affected by the conflict." A week ago, the Sudanese government and the Sudan People's Liberation Movement signed an agreement to allow peacekeepers in Abyei, aimed at ending strife that has ravaged much of the country.  The two sides agreed in principle on the need for a third party to monitor the ill-defined border between north and south before the scheduled July 9 independence for the south. The U.N. peacekeepers will "monitor and verify the redeployment of any Sudan Armed Forces, Sudan People's Liberation Army or its successor" from the Abyei area, among other tasks, the Security Council resolution states."""
EXAMPLE_ARTICLE_QUESTIONS = [
   "Did the U.N. Security Council approve a resolution to send 4,200 peacekeepers to Abyei, Sudan?",
   "Was there a recent agreement between Sudan and Southern Sudan?",
   "Does the resolution establish the UNISFA?",
   "Did the resolution pass unanimously?",
   "Did Hiliary Clinton commend the swift passage of the resolution?",
   "Has Abyei been a source of regional tension for many years?",
   "Were Sudan and Southern Sudan urged to reach an immediate cease-fire?"
   "Were Sudan and Southern Sudan urged to provide aid workers with unfettered access to deliver humanitarian assistance?"
   "Did the Sudanese government sign an agreement a week ago?",
   "Did the Sudanese governemnt allow peacekeepers in Abyei?",
   "Did the two parties agree in principal for a third party to monitor the ill-defined border?",
   "Is South Sudan scheduled for independence on July 9?",
   "Will U.N. peacekeepers monitor and verify the redeployment of Sudan Armed Forces, Sudan People's Liberation Army, and its successor?",
]

In [5]:
# Define the prompts for question generation and question answering

# First, we need to define some strings to insert into our prompts
question_json_schema_as_str = """{"questions": [< question 1 >, < question 2 >]})"""
answer_json_schema_as_str = """{"answer": < answer >}"""

def question_generation_system_prompt(num_questions, example_text, example_questions):
   """ Returns a tuple of system prompt and user message for question generation from a piece of text
   """
   example_questions_as_json_str = """{"questions": [""" + ", ".join(EXAMPLE_SUMMARY_QUESTIONS) + "}]"

   # We want to include a caveat with the example if the number of questions from the example differs from the number of questions we want to generate
   if len(example_questions) == num_questions:
      example_caveat = ""
   else:
      example_caveat = f"""There are {"more" if len(example_questions) > num_questions else "less"} than {num_questions} questions in the example, but the output must still have exactly {num_questions} questions. """
   
   system_prompt = f"""You are a super intelligent artificial intelligence that generate a perfectly compliant JSON with *exactly* {num_questions} close-ended questions based on text given from the user. The questions you generate can be answered with either a 'yes' or a 'no'. The schema for the JSON is {question_json_schema_as_str} where < question 1 > is the first question as a string < question 2 > is the second question as a string, etc.
Included is an example text and example question JSON output. {example_caveat}Do *not* make questions based on the example.
Do *not* output anything except the JSON. Do *not* output additional information, comments, or context

-- Instructions --
1) Think of a *strictly* close-ended question based on the text that can be answered with a 'yes' or 'no'
2) Make sure this question is different from all previous questions
3) Make sure the question is related to the given text and is NOT related to the example text
4) The correct answer to the question must be 'yes'
5) The question should be based *only* on the given text, and should not use any external information. The question *must* be able to be answered 'yes' using *only* the given text. Do *not* make up any information. Do *not* make any assumptions or inferences about the text
6) Add the question, and only the question to the JSON
7) Repeat steps 1 - 5 until *exactly* {num_questions} close-ended questions have been generated

--- Example Text ---
{example_text}

--- Example Output ---
{example_questions_as_json_str}

--- End of Example ---
"""
   
   return system_prompt

def question_generation_user_message(text):
   return text

def answer_generation_system_prompt():
   return f"""Based on the given text and close-ended question, answer that question with a 'yes', 'no', or 'idk'. The schema for the JSON is {answer_json_schema_as_str} where < answer > is a string of 'yes', 'no', or 'idk'
Answers should *strictly* be 'yes', 'no', or 'idk'
Do *not* output anything except the JSON. Do *not* output additional information, comments, or context

-- Instructions --
The answer should be 'yes' or 'no' if the given text contains the information relevant to answering the question
The answer should be 'idk' if the given text does not contain the information relevant to answering the question or if the answer is unknown or ambiguous based on the given text
*Only* use the given text to answer the question. Even if the correct answer is 'yes' or 'no', if the given text does not contain the answer, instead select 'idk'
*Only* output the answer ('yes', 'no', or 'idk') to the question, do not output any further information
"""

def answer_generation_user_message(text, question):
   return f"""
--- Given Text---
{text}

-- Question --
{question}
"""

#### Async Processing

We have 1600 pairs of summaries and articles to process, and the longest pole for each summary/article pair is the LLM call
In order to 

In [6]:
async def generate_qnas(llm_call_func, llm_payload_func, session, text, chars_per_q, example_text, example_questions):
    # Calculate the number of questions based on the given information density of the text
    num_questions = np.round(len(text) / chars_per_q, 0).astype(int).item()
    
    # Generate the questions
    # Note: sometimes the model returns an incorrect number of questions (sometimes its more, sometimes less)
    #   For more, we can just truncate the list after num_questions
    #   For less, we can generate a few extra in the hopes that the few extra will cancel out the model's mistake
    response = await llm_call_func(
        session,
        llm_payload_func(
            question_generation_system_prompt(num_questions + QUESTION_BUFFER, example_text, example_questions),
            question_generation_user_message(text),
            "questions",
            num_questions + QUESTION_BUFFER
            )
    )

    # Format the questions into a list with length num_questions (list of strings)
    content_as_json = json.loads(response)
    questions_as_list = content_as_json["questions"]
    questions_as_list = questions_as_list[:num_questions]

    for question in questions_as_list:
        if question is None:
            print("None")

    # Generate the answers
    #   The best performance comes when generating 1 answer per model call
    #   Giving it multiple questions at once tends to cause it to give more incorrect answers
    answer_generation_tasks = [
        llm_call_func(
            session,
            llm_payload_func(
                answer_generation_system_prompt(),
                answer_generation_user_message(text, question),
                "answer",
                )
        )
        for question in questions_as_list
    ]
    answers_as_json_strs = await asyncio.gather(*answer_generation_tasks)
    
    # Format the answers
    answers_as_list = []
    for answer_json_as_str in answers_as_json_strs:
        try:
            answers_as_list.append(json.loads(answer_json_as_str)["answer"])
        except:
            answers_as_list.append(None)
            print(answer_json_as_str)

    return pd.DataFrame({"questions":questions_as_list, "answers":answers_as_list})

In [7]:
question_sets = []

# The max number of llama instances I can fit in my GPT memory is 3
# I have a relatively low rate limit with Azure, so I will get rate limited before hitting 3 concurrent instances anyway
conn = aiohttp.TCPConnector(limit=3)

timeout = aiohttp.ClientTimeout(total=28800) # 8 hours
async with aiohttp.ClientSession(connector=conn, timeout=timeout) as session:
    summaries = summ_eval["machine_summary"][:100]
    articles = summ_eval["article"][:100]
        
    summary_generation_tasks = [
        generate_qnas(
            call_llama,
            llama_payload,
            session,
            summary,
            SUMMARY_CHARS_PER_Q,
            EXAMPLE_SUMMARY,
            EXAMPLE_SUMMARY_QUESTIONS
        )
        for summary in summaries
    ]
    summary_qna_sets = await asyncio.gather(*summary_generation_tasks)


CancelledError: 

In [ ]:
prompt = question_generation_message(summary["machine_summaries"][0][0])
AZURE_GPT_retval = {
    "messages" : [{
        "role": "system",
        "content": prompt
    }],
    "temperature": 0.0,
    "top_p": 0.0,
    "seed": 1234,
    "max_token": 10 + 40*NUMBER_OF_QUESTIONS,
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "output",
            "schema": {
                "type": "object",
                "properties": {
                    "question":{
                        "type": "array",
                        # "minItems": NUMBER_OF_QUESTIONS,
                        # "maxItems": NUMBER_OF_QUESTIONS,
                        "items":{
                            "type": "string",
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["questions"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
}

#### Citations

[1] Liu, Yang, et al. "G-eval: Nlg evaluation using gpt-4 with better human alignment." arXiv preprint arXiv:2303.16634 (2023).

[2] Fabbri, Alexander R., et al. "Summeval: Re-evaluating summarization evaluation." Transactions of the Association for Computational Linguistics 9 (2021): 391-409.

[3] Manakul, Potsawee, Adian Liusie, and Mark JF Gales. "MQAG: Multiple-choice question answering and generation for assessing information consistency in summarization." arXiv preprint arXiv:2301.12307 (2023).

[4] Confident AI Inc. (2024). The Open-Source LLM Evaluation Framework. DeepEval. https://docs.confident-ai.com/ 

[5] TruEra. (2024). TruLens for LLMs. TruLens. https://www.trulens.org/ 